# AKC-Tutor · Phase II — RoBERTa Emotion Recognition
**Fully automated Google Colab experiment** | Official GoEmotions simplified split | 28-class **single-label** classification | GPU training | Held-out evaluation | Baseline comparison | Results ZIP

**Run:** Colab → Runtime → Change runtime type → **GPU (T4/L4/A100)** → **Runtime → Run all**. Internet is required for dataset/model downloads. **No files need to be uploaded.**

**Methodological note:** This uses only examples with exactly one GoEmotions label, matching Phase I. Validation data selects the best epoch; the test partition is evaluated **once** after model selection. Phase I refitted on train+dev, whereas this transformer selects a train-only checkpoint: this is a useful reference comparison, but **not an identical training protocol**. No educational tutoring effectiveness is claimed from Reddit emotion data.

## 0 · Install dependencies
Use Colab's preinstalled NumPy, SciPy, PyTorch and scikit-learn to avoid the earlier NumPy binary mismatch. Install only Hugging Face packages. If Colab asks for a runtime restart, restart and run from the top.

In [ ]:
%pip -q install "transformers>=4.44,<5" "safetensors>=0.4"

## 1 · Configuration, GPU check and reproducibility

In [ ]:
import os, sys, json, time, random, gc, platform, urllib.request, shutil, zipfile
from pathlib import Path
from datetime import datetime, timezone
import numpy as np, pandas as pd, matplotlib.pyplot as plt
import sklearn, torch
from torch import nn
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import accuracy_score, f1_score, precision_recall_fscore_support, classification_report, confusion_matrix, ConfusionMatrixDisplay
from transformers import AutoTokenizer, AutoModelForSequenceClassification, get_linear_schedule_with_warmup

SEED = 42
MODEL_NAME = 'FacebookAI/roberta-base'
MAX_LENGTH = 128
BATCH_SIZE = 16          # reduce to 8 if GPU out of memory
GRAD_ACCUM = 2           # effective batch size 32
EPOCHS = 3
LEARNING_RATE = 2e-5
WEIGHT_DECAY = 0.01
WARMUP_RATIO = 0.10
PATIENCE = 2
NUM_WORKERS = 2
BOOTSTRAP_REPS = 1000
SAVE_MODEL_CHECKPOINT = False  # True to save ~500MB model in output directory (not in results ZIP)
OUTPUT = Path('/content/AKC_Tutor_Phase2') if Path('/content').exists() else Path('AKC_Tutor_Phase2')
OUTPUT.mkdir(parents=True, exist_ok=True)
DATA = OUTPUT/'dataset'; DATA.mkdir(exist_ok=True)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
assert DEVICE.type == 'cuda', 'GPU not detected. Colab → Runtime → Change runtime type → T4/L4/A100 GPU, then reconnect.'
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False
print('Python:',sys.version.split()[0], '| torch:',torch.__version__, '| sklearn:',sklearn.__version__)
print('GPU:',torch.cuda.get_device_name(0), '| VRAM GB:',round(torch.cuda.get_device_properties(0).total_memory/1e9,2))
print('Output:',OUTPUT)


## 2 · Automatically download the official GoEmotions simplified dataset

In [ ]:
BASE='https://raw.githubusercontent.com/google-research/google-research/master/goemotions/data/'
for filename in ['train.tsv','dev.tsv','test.tsv','emotions.txt']:
    dest=DATA/filename
    if dest.exists() and dest.stat().st_size>0:
        print('Cached:',filename); continue
    err=None
    for attempt in range(4):
        try:
            req=urllib.request.Request(BASE+filename,headers={'User-Agent':'Mozilla/5.0'})
            with urllib.request.urlopen(req,timeout=60) as response, open(dest,'wb') as out:
                shutil.copyfileobj(response,out)
            print('Downloaded:',filename,dest.stat().st_size,'bytes');err=None;break
        except Exception as e:
            err=e;time.sleep(2*(attempt+1))
    if err:raise RuntimeError(f'Failed to download {filename}: {err}')
labels=[x.strip() for x in (DATA/'emotions.txt').read_text().splitlines() if x.strip()]
assert len(labels)==28, f'Expected 28 labels, got {len(labels)}'
print('Labels:',labels)


## 3 · Parse and validate single-label train/dev/test partitions

In [ ]:
def read_single_label(name):
    df=pd.read_csv(DATA/f'{name}.tsv',sep='\t',header=None,names=['text','label_ids','comment_id'],dtype=str,keep_default_na=False)
    df['ids']=df.label_ids.map(lambda s:[int(v.strip()) for v in s.split(',') if v.strip()])
    assert df.ids.map(lambda a:all(0<=v<len(labels) for v in a)).all()
    original=len(df)
    df=df.loc[(df.ids.map(len)==1)&(df.text.str.strip()!='')].copy()
    df['label']=df.ids.map(lambda a:a[0]);df=df[['text','label','comment_id']].reset_index(drop=True)
    print(f'{name}: {original:,} total → {len(df):,} single-label records')
    return df
train=read_single_label('train'); dev=read_single_label('dev'); test=read_single_label('test')
assert len(train)==36308 and len(dev)==4548 and len(test)==4590, 'Split sizes differ from Phase I; inspect input parsing before continuing.'
assert train.label.between(0,27).all() and dev.label.between(0,27).all() and test.label.between(0,27).all()
counts=pd.DataFrame({'emotion':labels,'train':train.label.value_counts().reindex(range(28),fill_value=0).values,'dev':dev.label.value_counts().reindex(range(28),fill_value=0).values,'test':test.label.value_counts().reindex(range(28),fill_value=0).values})
counts.to_csv(OUTPUT/'class_distribution.csv',index=False)
print(counts.to_string(index=False))


## 4 · Tokenize and construct efficient PyTorch data loaders

In [ ]:
tokenizer=AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True)
class EmotionDataset(Dataset):
    def __init__(self, frame):
        self.text=frame.text.tolist();self.y=frame.label.astype(int).tolist()
    def __len__(self):return len(self.y)
    def __getitem__(self,idx):return self.text[idx],self.y[idx]
def collate(batch):
    texts,ys=zip(*batch)
    tokens=tokenizer(list(texts),padding=True,truncation=True,max_length=MAX_LENGTH,return_tensors='pt')
    tokens['labels']=torch.tensor(ys,dtype=torch.long)
    return tokens
train_loader=DataLoader(EmotionDataset(train),batch_size=BATCH_SIZE,shuffle=True,collate_fn=collate,num_workers=NUM_WORKERS,pin_memory=True)
dev_loader=DataLoader(EmotionDataset(dev),batch_size=BATCH_SIZE*2,shuffle=False,collate_fn=collate,num_workers=NUM_WORKERS,pin_memory=True)
test_loader=DataLoader(EmotionDataset(test),batch_size=BATCH_SIZE*2,shuffle=False,collate_fn=collate,num_workers=NUM_WORKERS,pin_memory=True)
print('Train batches:',len(train_loader),'Validation batches:',len(dev_loader))


## 5 · Fine-tune RoBERTa on GPU (best epoch selected by validation Macro-F1)

In [ ]:
model=AutoModelForSequenceClassification.from_pretrained(MODEL_NAME,num_labels=len(labels),id2label={i:x for i,x in enumerate(labels)},label2id={x:i for i,x in enumerate(labels)}).to(DEVICE)
optimizer=torch.optim.AdamW(model.parameters(),lr=LEARNING_RATE,weight_decay=WEIGHT_DECAY)
updates_per_epoch=(len(train_loader)+GRAD_ACCUM-1)//GRAD_ACCUM
num_steps=updates_per_epoch*EPOCHS
scheduler=get_linear_schedule_with_warmup(optimizer,num_warmup_steps=int(num_steps*WARMUP_RATIO),num_training_steps=num_steps)
use_amp=DEVICE.type=='cuda';scaler=torch.amp.GradScaler('cuda',enabled=use_amp)

def infer(loader):
    model.eval(); y_all=[];p_all=[];scores_all=[]
    with torch.inference_mode():
        for batch in loader:
            y=batch.pop('labels').numpy()
            batch={k:v.to(DEVICE,non_blocking=True) for k,v in batch.items()}
            with torch.autocast(device_type='cuda',dtype=torch.float16,enabled=use_amp):
                logits=model(**batch).logits
            scores=torch.softmax(logits.float(),dim=-1).cpu().numpy()
            y_all.extend(y.tolist());p_all.extend(scores.argmax(axis=1).tolist());scores_all.append(scores)
    return np.array(y_all),np.array(p_all),np.concatenate(scores_all)

history=[];best_val=-1.;best_epoch=0;bad_epochs=0
best_path=OUTPUT/'best_roberta_weights.pt'
for epoch in range(1,EPOCHS+1):
    model.train();optimizer.zero_grad(set_to_none=True);running_loss=0.;start=time.time()
    for step,batch in enumerate(train_loader):
        batch={k:v.to(DEVICE,non_blocking=True) for k,v in batch.items()}
        with torch.autocast(device_type='cuda',dtype=torch.float16,enabled=use_amp):
            loss=model(**batch).loss
        running_loss+=float(loss.detach().item())
        scaler.scale(loss/GRAD_ACCUM).backward()
        if (step+1)%GRAD_ACCUM==0 or (step+1)==len(train_loader):
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(),1.0)
            scaler.step(optimizer);scaler.update();optimizer.zero_grad(set_to_none=True);scheduler.step()
        if (step+1)%200==0:print(f'Epoch {epoch}/{EPOCHS}, batch {step+1}/{len(train_loader)}',flush=True)
    val_true,val_pred,_=infer(dev_loader)
    val_f1=f1_score(val_true,val_pred,labels=list(range(28)),average='macro',zero_division=0)
    row={'epoch':epoch,'train_loss':running_loss/len(train_loader),'validation_macro_f1':float(val_f1),'seconds':round(time.time()-start,1)}
    history.append(row);print(row,flush=True)
    if val_f1>best_val:
        best_val=val_f1;best_epoch=epoch;bad_epochs=0
        torch.save(model.state_dict(),best_path)
        print('Saved best validation checkpoint')
    else:
        bad_epochs+=1
        if bad_epochs>=PATIENCE:
            print('Early stopping');break
pd.DataFrame(history).to_csv(OUTPUT/'training_history.csv',index=False)
model.load_state_dict(torch.load(best_path,map_location=DEVICE,weights_only=True))
print('Selected epoch:',best_epoch,'validation Macro-F1:',round(best_val,4))


## 6 · One-time held-out test evaluation and per-class metrics

In [ ]:
y_true,y_pred,prob=infer(test_loader)
assert len(y_true)==len(test) and np.array_equal(y_true,test.label.to_numpy())
ids=list(range(28))
report=classification_report(y_true,y_pred,labels=ids,target_names=labels,output_dict=True,zero_division=0)
metrics={'accuracy':float(accuracy_score(y_true,y_pred)),'macro_f1':float(f1_score(y_true,y_pred,labels=ids,average='macro',zero_division=0)),'weighted_f1':float(f1_score(y_true,y_pred,labels=ids,average='weighted',zero_division=0)),'n_train':len(train),'n_dev':len(dev),'n_test':len(test),'best_epoch':best_epoch,'best_validation_macro_f1':float(best_val),'seed':SEED,'model':MODEL_NAME,'label_count':28,'subset':'single-label official simplified GoEmotions','test_used_for_tuning':False}
per_class=pd.DataFrame([{'emotion':name,**report[name]} for name in labels]);per_class.to_csv(OUTPUT/'classification_report.csv',index=False)
preds=test[['comment_id','text']].copy();preds['true_label_id']=y_true;preds['pred_label_id']=y_pred;preds['true_emotion']=[labels[i] for i in y_true];preds['pred_emotion']=[labels[i] for i in y_pred];preds['confidence']=prob.max(axis=1)
preds.to_csv(OUTPUT/'test_predictions.csv',index=False)
print(json.dumps(metrics,indent=2));print(per_class.to_string(index=False))


## 7 · Bootstrap 95% confidence intervals (test set resampling)

In [ ]:
rng=np.random.default_rng(SEED); boot=np.empty(BOOTSTRAP_REPS)
for b in range(BOOTSTRAP_REPS):
    ix=rng.integers(0,len(y_true),size=len(y_true))
    boot[b]=f1_score(y_true[ix],y_pred[ix],labels=ids,average='macro',zero_division=0)
ci=np.quantile(boot,[.025,.975]);metrics['macro_f1_bootstrap_95ci']=[float(ci[0]),float(ci[1])];metrics['bootstrap_replicates']=BOOTSTRAP_REPS
fig,ax=plt.subplots(figsize=(7,4));ax.hist(boot,bins=35);ax.axvline(ci[0],linestyle='--');ax.axvline(ci[1],linestyle='--');ax.set(xlabel='Bootstrap Macro-F1',ylabel='Frequency',title='RoBERTa: test Macro-F1 bootstrap interval');fig.tight_layout();fig.savefig(OUTPUT/'macro_f1_bootstrap.png',dpi=160);plt.show();plt.close(fig)
print('Macro-F1:',round(metrics['macro_f1'],4),'95% bootstrap CI:',np.round(ci,4))


## 8 · Confusion matrix and training curves

In [ ]:
cm=confusion_matrix(y_true,y_pred,labels=ids);pd.DataFrame(cm,index=labels,columns=labels).to_csv(OUTPUT/'confusion_matrix.csv')
fig,ax=plt.subplots(figsize=(15,13));ConfusionMatrixDisplay(cm,display_labels=labels).plot(ax=ax,xticks_rotation=90,values_format='d',colorbar=False);ax.set_title('RoBERTa: GoEmotions test confusion matrix');fig.tight_layout();fig.savefig(OUTPUT/'confusion_matrix.png',dpi=160);plt.show();plt.close(fig)
h=pd.DataFrame(history);fig,ax=plt.subplots(figsize=(7,4));ax.plot(h.epoch,h.train_loss,marker='o');ax.set(xlabel='Epoch',ylabel='Mean training loss',title='Training loss');fig.tight_layout();fig.savefig(OUTPUT/'training_loss.png',dpi=160);plt.show();plt.close(fig)
fig,ax=plt.subplots(figsize=(7,4));ax.plot(h.epoch,h.validation_macro_f1,marker='o');ax.set(xlabel='Epoch',ylabel='Validation Macro-F1',title='Validation performance');fig.tight_layout();fig.savefig(OUTPUT/'validation_macro_f1.png',dpi=160);plt.show();plt.close(fig)


## 9 · Phase I baseline comparison
Reference baseline values are taken from the **previously uploaded Phase I results ZIP** (single-label GoEmotions, 4,590 test samples): Accuracy **0.5618736383**, Macro-F1 **0.3735847443**, Weighted-F1 **0.5267252477**. These values are **historical measured results**, not generated by this notebook. The comparison is descriptive because Phase I refitted on train+dev and Phase II selects a train-only checkpoint. For **paired** statistical comparisons, optionally upload the Phase I `test_predictions.csv` or ZIP in the next cell.

In [ ]:
baseline={'accuracy':0.5618736383442265,'macro_f1':0.37358474434775174,'weighted_f1':0.5267252476916996}
comparison=pd.DataFrame([{'model':'Phase I TF-IDF + Logistic Regression',**baseline},{'model':'Phase II RoBERTa',**{k:metrics[k] for k in baseline}}]);comparison.to_csv(OUTPUT/'baseline_comparison.csv',index=False)
print(comparison.to_string(index=False))
fig,ax=plt.subplots(figsize=(8,4));xx=np.arange(3);w=.35
ax.bar(xx-w/2,[baseline[k] for k in baseline],w,label='TF-IDF + Logistic Regression')
ax.bar(xx+w/2,[metrics[k] for k in baseline],w,label='RoBERTa')
ax.set_xticks(xx,list(baseline.keys()));ax.set_ylim(0,1);ax.set_ylabel('Score');ax.legend();ax.set_title('Phase I vs Phase II (descriptive comparison)');fig.tight_layout();fig.savefig(OUTPUT/'baseline_comparison.png',dpi=160);plt.show();plt.close(fig)


## 10 · Optional paired comparison with Phase I predictions
Leave `UPLOAD_PHASE1_PREDICTIONS = False` for a fully automatic run. If set to `True`, Colab will ask for the previous Phase I ZIP or `test_predictions.csv`. It then aligns by `comment_id` **and** true label and computes paired bootstrap difference in Macro-F1. Do not assume paired comparison if the files do not align.

In [ ]:
UPLOAD_PHASE1_PREDICTIONS=False
if UPLOAD_PHASE1_PREDICTIONS:
    from google.colab import files
    uploaded=files.upload()
    name=next(iter(uploaded)); raw=uploaded[name]
    if name.lower().endswith('.zip'):
        import io
        with zipfile.ZipFile(io.BytesIO(raw)) as z:
            matches=[p for p in z.namelist() if p.endswith('test_predictions.csv')]
            assert len(matches)==1, 'Expected exactly one test_predictions.csv in ZIP'
            old=pd.read_csv(z.open(matches[0]))
    else:old=pd.read_csv(name)
    print('Phase I prediction columns:',list(old.columns))
    assert 'comment_id' in old.columns and 'comment_id' in preds.columns
    assert old.comment_id.is_unique and preds.comment_id.is_unique, 'IDs are not unique; cannot safely pair'
    joined=preds.merge(old,on='comment_id',suffixes=('_phase2','_phase1'),validate='one_to_one')
    assert len(joined)==len(preds)==len(old), 'Phase I and II test rows differ'
    # Phase I notebook uses true_label_id/pred_label_id or true_id/pred_id depending on version
    old_true=next((c for c in ['true_label_id_phase1','true_id','y_true','true_label'] if c in joined),None)
    old_pred=next((c for c in ['pred_label_id_phase1','pred_id','y_pred','predicted_label'] if c in joined),None)
    assert old_true is not None and old_pred is not None, 'Could not identify Phase I numeric label columns'
    assert np.array_equal(joined.true_label_id_phase2.to_numpy(),joined[old_true].to_numpy()), 'True labels mismatch'
    a=joined[old_pred].to_numpy().astype(int);b=joined.pred_label_id_phase2.to_numpy().astype(int);yt=joined.true_label_id_phase2.to_numpy().astype(int)
    rng=np.random.default_rng(SEED);diffs=[]
    for _ in range(BOOTSTRAP_REPS):
        ix=rng.integers(0,len(yt),len(yt))
        diffs.append(f1_score(yt[ix],b[ix],labels=ids,average='macro',zero_division=0)-f1_score(yt[ix],a[ix],labels=ids,average='macro',zero_division=0))
    paired={'observed_macro_f1_difference':float(f1_score(yt,b,labels=ids,average='macro',zero_division=0)-f1_score(yt,a,labels=ids,average='macro',zero_division=0)),'paired_bootstrap_difference_95ci':np.quantile(diffs,[.025,.975]).tolist(),'bootstrap_replicates':BOOTSTRAP_REPS}
    (OUTPUT/'paired_comparison.json').write_text(json.dumps(paired,indent=2))
    print(paired)
else:print('Paired comparison skipped; descriptive baseline comparison is included.')


## 11 · Export all reproducibility metadata and download results ZIP

In [ ]:
metrics_path=OUTPUT/'test_metrics.json';metrics_path.write_text(json.dumps(metrics,indent=2))
metadata={'timestamp_utc':datetime.now(timezone.utc).isoformat(),'python':sys.version,'torch':torch.__version__,'sklearn':sklearn.__version__,'gpu':torch.cuda.get_device_name(0),'model':MODEL_NAME,'max_length':MAX_LENGTH,'batch_size':BATCH_SIZE,'gradient_accumulation':GRAD_ACCUM,'epochs_requested':EPOCHS,'best_epoch':best_epoch,'learning_rate':LEARNING_RATE,'weight_decay':WEIGHT_DECAY,'warmup_ratio':WARMUP_RATIO,'seed':SEED,'dataset_url':BASE,'splits':{'train':len(train),'dev':len(dev),'test':len(test)},'protocol':'Train-only fine-tuning; select best checkpoint by validation Macro-F1; held-out test once','baseline_protocol':'Phase I TF-IDF refitted on train+dev after validation C selection','limitations':['Reddit-derived emotions; not validated on educational dialogue','Single-label subset excludes multi-label examples','No knowledge state or conversational tutoring evaluated in Phase II']}
(OUTPUT/'run_metadata.json').write_text(json.dumps(metadata,indent=2))
(OUTPUT/'README.txt').write_text('AKC-Tutor Phase II: RoBERTa GoEmotions single-label test results.\nTrain/dev/test official partitions; 28 classes.\nBest epoch selected on dev; test evaluated once.\nPhase I comparison is descriptive unless paired_comparison.json exists.\nModel weights not included in results ZIP.\n')
if SAVE_MODEL_CHECKPOINT:
    model.save_pretrained(OUTPUT/'trained_roberta',safe_serialization=True)
    tokenizer.save_pretrained(OUTPUT/'trained_roberta')
    print('Model checkpoint saved at:',OUTPUT/'trained_roberta','(not in results ZIP)')
zip_path=Path('/content/AKC_Tutor_Phase2_Results.zip') if Path('/content').exists() else Path('AKC_Tutor_Phase2_Results.zip')
with zipfile.ZipFile(zip_path,'w',compression=zipfile.ZIP_DEFLATED,compresslevel=6) as z:
    for p in sorted(OUTPUT.iterdir()):
        if p.is_file() and p.suffix.lower() in {'.json','.csv','.png','.txt'}:
            z.write(p,arcname=p.name)
print('RESULTS ZIP:',zip_path,'size MB:',round(zip_path.stat().st_size/1e6,2))
print('Final test scores:',{k:round(metrics[k],4) for k in ['accuracy','macro_f1','weighted_f1']})
try:
    from google.colab import files
    files.download(str(zip_path))
except ImportError:
    print('Not in Colab. Download manually from:',zip_path)
